# AVO intercept-gradient analysis

This notebook calculates the **intercept** and **gradient** needed for an AVO crossplot using the dataset `DataAVO2.csv`.

Only these two interfaces are evaluated, as requested:
- shale to brine
- shale to gas

The dataset contains a broken shale-density column (`Shale_Vs.1` is a duplicate of `Brine_Vs`). To keep the workflow reproducible, shale density is estimated from shale P-wave velocity with the Gardner relation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.patches import Rectangle

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')


In [ ]:
data_path = '../Datasets/DataAVO2.csv'
df = pd.read_csv(data_path).rename(columns={'Shale_VP': 'Shale_Vp'})

def gardner_density(vp_km_s: pd.Series) -> pd.Series:
    """Estimate density [g/cc] from Vp [km/s] using Gardner after converting to m/s."""
    vp_m_s = vp_km_s * 1000.0
    return 0.31 * vp_m_s ** 0.25

df['Shale_rho'] = gardner_density(df['Shale_Vp'])

display(df[['Index', 'Shale_Vp', 'Shale_Vs', 'Shale_rho', 'Brine_Vp', 'Brine_Vs', 'Brine_p', 'Gas_Vp', 'Gas_Vs', 'Gas_p']].head())
print('Note: shale density is estimated with Gardner because the shale-density column in the CSV is corrupted.')


In [ ]:
def shuey_intercept_gradient(vp1, vs1, rho1, vp2, vs2, rho2):
    """Return Shuey two-term intercept (A) and gradient (B)."""
    vp_avg = 0.5 * (vp1 + vp2)
    vs_avg = 0.5 * (vs1 + vs2)
    rho_avg = 0.5 * (rho1 + rho2)

    d_vp = vp2 - vp1
    d_vs = vs2 - vs1
    d_rho = rho2 - rho1

    intercept = 0.5 * (d_vp / vp_avg + d_rho / rho_avg)
    gradient = 0.5 * (d_vp / vp_avg) - 2.0 * (vs_avg / vp_avg) ** 2 * (2.0 * d_vs / vs_avg + d_rho / rho_avg)
    return intercept, gradient

def classify_avo(intercept, gradient, near_zero=0.02):
    if abs(intercept) <= near_zero and gradient < 0:
        return 'Class II'
    if intercept > 0 and gradient < 0:
        return 'Class I'
    if intercept < 0 and gradient < 0:
        return 'Class III'
    if intercept < 0 and gradient > 0:
        return 'Class IV'
    return 'Unclassified'

def build_interface_table(frame, interface_name, vp_col, vs_col, rho_col, angles=np.arange(0, 41, 5)):
    records = []
    for row in frame.itertuples(index=False):
        intercept, gradient = shuey_intercept_gradient(
            row.Shale_Vp, row.Shale_Vs, row.Shale_rho,
            getattr(row, vp_col), getattr(row, vs_col), getattr(row, rho_col)
        )
        reflectivity = intercept + gradient * np.sin(np.radians(angles)) ** 2
        records.append({
            'Index': row.Index,
            'Interface': interface_name,
            'Intercept': intercept,
            'Gradient': gradient,
            'AVO_Class': classify_avo(intercept, gradient),
            'R(0deg)': reflectivity[0],
            'R(20deg)': reflectivity[np.where(angles == 20)[0][0]],
            'R(40deg)': reflectivity[-1],
        })
    return pd.DataFrame(records)

brine_results = build_interface_table(df, 'Shale to Brine', 'Brine_Vp', 'Brine_Vs', 'Brine_p')
gas_results = build_interface_table(df, 'Shale to Gas', 'Gas_Vp', 'Gas_Vs', 'Gas_p')
avo_results = pd.concat([brine_results, gas_results], ignore_index=True)

display(avo_results.head(10))


In [ ]:
summary = (
    avo_results.groupby(['Interface', 'AVO_Class'])[['Intercept', 'Gradient']]
    .agg(['count', 'mean'])
    .round(4)
)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

angles = np.arange(0, 41, 5)
interface_styles = {
    'Shale to Brine': {'color': '#1f77b4', 'marker': 'o'},
    'Shale to Gas': {'color': '#d62728', 'marker': '^'},
}

# Left panel: AVO curves generated from the intercept and gradient
for interface_name, subset in avo_results.groupby('Interface'):
    style = interface_styles[interface_name]
    for row in subset.itertuples(index=False):
        amplitude = row.Intercept + row.Gradient * np.sin(np.radians(angles)) ** 2
        axes[0].plot(angles, amplitude, color=style['color'], alpha=0.25)

axes[0].set_title('Shuey two-term reflectivity curves')
axes[0].set_xlabel('Incident angle (degrees)')
axes[0].set_ylabel('Reflection coefficient')
axes[0].axhline(0, color='black', linewidth=1)

# Right panel: intercept-gradient crossplot with AVO class areas
x = avo_results['Intercept']
y = avo_results['Gradient']
x_pad = max(0.02, 0.15 * (x.max() - x.min()))
y_pad = max(0.02, 0.15 * (y.max() - y.min()))
xmin, xmax = x.min() - x_pad, x.max() + x_pad
ymin, ymax = y.min() - y_pad, y.max() + y_pad
class_ii_band = max(0.02, 0.12 * (xmax - xmin))

ax = axes[1]
ax.add_patch(Rectangle((0, ymin), xmax, -ymin, facecolor='#d6ebff', alpha=0.35, zorder=0))
ax.add_patch(Rectangle((xmin, ymin), -xmin, -ymin, facecolor='#ffd9d2', alpha=0.35, zorder=0))
ax.add_patch(Rectangle((xmin, 0), -xmin, ymax, facecolor='#d8f3dc', alpha=0.45, zorder=0))
ax.add_patch(Rectangle((-class_ii_band / 2, ymin), class_ii_band, -ymin, facecolor='#fff3bf', alpha=0.7, zorder=0))

for interface_name, subset in avo_results.groupby('Interface'):
    style = interface_styles[interface_name]
    ax.scatter(
        subset['Intercept'], subset['Gradient'],
        s=70, marker=style['marker'], c=style['color'],
        edgecolor='black', linewidth=0.5, alpha=0.9, label=interface_name
    )
    for row in subset.itertuples(index=False):
        ax.annotate(int(row.Index), (row.Intercept, row.Gradient), xytext=(4, 4), textcoords='offset points', fontsize=8, alpha=0.8)

ax.axhline(0, color='black', linewidth=1)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_title('Intercept-gradient crossplot')
ax.set_xlabel('Intercept, A')
ax.set_ylabel('Gradient, B')
ax.legend(loc='upper right')

ax.text(0.68 * xmax, 0.78 * ymin, 'Class I', fontsize=11, weight='bold', color='#22577a')
ax.text(0.0, 0.82 * ymin, 'Class II', fontsize=11, weight='bold', color='#8a5a00', ha='center')
ax.text(0.72 * xmin, 0.78 * ymin, 'Class III', fontsize=11, weight='bold', color='#9d0208')
ax.text(0.72 * xmin, 0.78 * ymax, 'Class IV', fontsize=11, weight='bold', color='#2d6a4f')

plt.show()


In [ ]:
avo_results.sort_values(['Interface', 'Index']).reset_index(drop=True)
